In [1]:
from copy import deepcopy

import fasttext
import fasttext.util
import matplotlib.pyplot as plt
from matplotlib.image import imread
from mpl_toolkits import mplot3d
from matplotlib import gridspec
from PIL import Image
import io
import os
from urllib.request import urlopen
from skimage.segmentation import mark_boundaries
from nltk.tokenize import RegexpTokenizer
from torchinfo import summary
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd
import requests
from scipy.stats import norm
import torch

from sklearn.metrics import classification_report
from torch.utils.tensorboard import SummaryWriter

from torchvision import datasets, transforms

2026-03-05 20:07:30.032308: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

## Код для обучения

In [4]:
class callback():
    def __init__(self, writer, dataset, loss_function, delimeter = 100, batch_size=64):
        self.step = 0
        self.writer = writer
        self.delimeter = delimeter
        self.loss_function = loss_function
        self.batch_size = batch_size

        self.dataset = dataset

    def forward(self, model, loss):
        self.step += 1
        self.writer.add_scalar('LOSS/train', loss, self.step)
        
        if self.step % self.delimeter == 0:
            
            batch_generator = torch.utils.data.DataLoader(dataset = self.dataset, 
                                                          batch_size=self.batch_size, shuffle=True)
            
            pred = []
            real = []
            test_loss = 0
            model.eval()

            for it, (x_batch, y_batch) in enumerate(batch_generator):
                x_batch = x_batch.to(model.device)
                y_batch = y_batch.to(model.device)

                output = model(x_batch)

                test_loss += self.loss_function(output, y_batch).cpu().item() * len(x_batch)
            
            test_loss /= len(self.dataset)
            
            self.writer.add_scalar('LOSS/test', test_loss, self.step)
          
    def __call__(self, model, loss):
        return self.forward(model, loss)

In [5]:
def train_on_batch(model, x_batch, y_batch, optimizer, loss_function):
    model.train()
    optimizer.zero_grad()
    
    output = model(x_batch.to(model.device))
    
    loss = loss_function(output, y_batch.to(model.device))
    loss.backward()

    optimizer.step()
    return loss.cpu().item()

In [6]:
def train_epoch(train_generator, model, loss_function, optimizer, callback = None):
    epoch_loss = 0
    total = 0
    for it, (batch_of_x, batch_of_y) in enumerate(train_generator):
        batch_loss = train_on_batch(model, batch_of_x, batch_of_y, optimizer, loss_function)
        
        if callback is not None:
            with torch.no_grad():
                callback(model, batch_loss)
            
        epoch_loss += batch_loss*len(batch_of_x)
        total += len(batch_of_x)
    
    return epoch_loss/total

In [7]:
def trainer(count_of_epoch, 
            batch_size, 
            dataset,
            model, 
            loss_function,
            optimizer,
            lr = 0.001,
            callback = None):

    optima = optimizer(model.parameters(), lr=lr)
    
    iterations = tqdm(range(count_of_epoch), desc='epoch')
    iterations.set_postfix({'train epoch loss': np.nan})
    for it in iterations:
        batch_generator = tqdm(
            torch.utils.data.DataLoader(dataset=dataset, 
                                        batch_size=batch_size, 
                                        shuffle=True, pin_memory=True), 
            leave=False, total=len(dataset)//batch_size+(len(dataset)%batch_size>0))
        
        epoch_loss = train_epoch(train_generator=batch_generator, 
                    model=model, 
                    loss_function=loss_function, 
                    optimizer=optima, 
                    callback=callback)
        
        iterations.set_postfix({'train epoch loss': epoch_loss})

In [8]:
def testing_on_test_sample(model, dataset_test_pt):
    batch_generator = torch.utils.data.DataLoader(dataset=dataset_test_pt, 
                                                  batch_size=64, 
                                                  pin_memory=True)
                
    pred = []
    real = []
    model.eval()
    for it, (x_batch, y_batch) in enumerate(batch_generator):
        x_batch = x_batch.to(device)
        with torch.no_grad():
            output = model(x_batch)
    
        preds = torch.argmax(output, dim=1)
    
        pred.extend(preds.cpu().numpy().flatten())
        real.extend(y_batch.cpu().numpy().flatten())
    
    print(classification_report(real, pred))

## Загрузка датасета

In [9]:
!pip install conllu

In [10]:
import gzip
from conllu import parse_incr

dataset = []
dataset_len = 1000000
upos_to_ind = {'[PAD]': 0}

with open('nerus_lenta.conllu', 'rt', encoding='utf-8') as f:
    count = 0
    for sent in parse_incr(f):
        text = sent.metadata['text']
        dataset.append([text, []]) # текст и ответ - размеченые части речи
        for token in sent:
            if token['upos'] not in upos_to_ind:
                upos_to_ind[token['upos']] = upos_to_ind.__len__()
            dataset[len(dataset) - 1][1].append(upos_to_ind[token['upos']])
        count += 1
        if count >= dataset_len:
            break

In [11]:
import psutil

# Получаем статистику памяти
mem = psutil.virtual_memory()

print(f"Всего: {mem.total / (1024**3):.2f} GB")
print(f"Доступно: {mem.available / (1024**3):.2f} GB")
print(f"Используется: {mem.used / (1024**3):.2f} GB")
print(f"Процент использования: {mem.percent}%")

Всего: 15.43 GB
Доступно: 8.27 GB
Используется: 7.17 GB
Процент использования: 46.4%


In [19]:
print(dataset[:1])
print(len(upos_to_ind))

[['Вице-премьер по социальным вопросам Татьяна Голикова рассказала, в каких регионах России зафиксирована наиболее высокая смертность от рака, сообщает РИА Новости.', [1, 2, 3, 1, 4, 4, 5, 6, 2, 7, 1, 4, 5, 8, 3, 1, 2, 1, 6, 5, 4, 4, 6]]]
18


#### Итого у нас 17 частей речи есть датасет, вида [предложение, [части речи слов]]

In [12]:
from sklearn.model_selection import train_test_split

dataset_train, dataset_test = train_test_split(dataset, test_size=0.2, random_state=42)

In [21]:
word_to_ind = {'[PAD]': 0, '[UNK]': 1, '[CLS]': 2, '[SEP]': 3}
for item in tqdm(dataset_train):
    sent = item[0]
    for word in RegexpTokenizer('[a-zA-Z]+|[^\w\s]|\d+').tokenize(sent):
        if word not in word_to_ind:
            word_to_ind[word] = word_to_ind.__len__()

  0%|          | 0/800000 [00:00<?, ?it/s]

In [22]:
print(len(word_to_ind))

43515


In [18]:
max_length_sentence = 15 # будем образать хороший вопрос насколько это коректно

In [24]:
class Tokenizer(object):
    def __init__(self, word_to_ind, tokenizer):
        self.word_to_ind = word_to_ind
        self.tokenizer = tokenizer
    def __call__(self, sentences, max_length = max_length_sentence - 2, pad_to_max_length = False): # -2 так как не учитываем ['[SEP]'] [['[CLS]']
        tokens = self.tokenizer.tokenize_sents(sentences)
        if not pad_to_max_length: # как я понимаю это приколы с максимальной размерностью предkj;tybz
            max_length = min(max_length, max(map(len, tokens)))
        tokens = [['[CLS]']+s+['[SEP]'] + ['[PAD]']*(max_length-len(s)) \
                  if len(s) < max_length \
                  else ['[CLS]']+s[:max_length]+['[SEP]'] \
                  for s in tokens ]
        ids = [[self.word_to_ind.get(w, self.word_to_ind['[UNK]']) for w in sent] for sent in tokens]
        return torch.tensor(ids)

In [ ]:
tokenizer = Tokenizer(word_to_ind, RegexpTokenizer('[a-zA-Z]+|[^\w\s]|\d+'))

In [26]:
train_texts = [item[0] for item in dataset_train]
test_texts  = [item[0] for item in dataset_test]

train_data_sent = tokenizer(train_texts)
test_data_sent  = tokenizer(test_texts)

In [27]:
train_y = []
test_y  = []

for item in dataset_train:
    if (len(item[1]) < max_length_sentence):
        item[1].extend([0] * (max_length_sentence - len(item[1]))) # добовляем падинг!
    train_y.append(item[1][:15])

for item in dataset_test:
    if (len(item[1]) < max_length_sentence):
        item[1].extend([0] * (max_length_sentence - len(item[1]))) # добовляем падинг!
    test_y.append(item[1][:15])

In [28]:
dataset_train_pt = torch.utils.data.TensorDataset(
    train_data_sent, torch.tensor(train_y))
dataset_test_pt = torch.utils.data.TensorDataset(
    test_data_sent, torch.tensor(test_y))

## Создаём Rnn модель

In [13]:
class RNN_1(torch.nn.Module):
    @property
    def device(self):
        return next(self.parameters()).device
    def __init__(self, vocab_dim, output_dim, num_tags = 18, emb_dim = 10, hidden_dim = 10, 
                 num_layers = 3, bidirectional = False, p=0.7):
        super(RNN_1, self).__init__()
        self.num_tags = num_tags
        self.vocab_dim = vocab_dim
        self.embedding = torch.nn.Embedding(vocab_dim, emb_dim)
        self.encoder = torch.nn.LSTM(emb_dim, hidden_dim, num_layers, 
                                     bidirectional=bidirectional, 
                                     batch_first=True, dropout=p)
        self.linear = torch.nn.Linear(
            2*num_layers*int(bidirectional + 1)*hidden_dim, 
            output_dim)
    def forward(self, input):
        input = self.embedding(input)
        _, (h, c) = self.encoder(input)
        act = torch.cat([h, c], dim=0).transpose(0, 1)
        act = act.reshape(len(input), -1)
        act = self.linear(act)

        batch_size_actual = input.shape[0]
        act = act.view(batch_size_actual, num_classes, max_length_sentence)
        return act

 ### итак у нас на вход числа - закодированые слова их max_length_sentence. На выходе выдадим матрицу max_length_sentence * len(upos_to_ind) - для каждлого слова ветор логитов принадлежнгости к классу.

In [17]:
num_classes = len(upos_to_ind)

In [42]:
import GPUtil; [print(f'GPU {g.id}: {g.memoryUsed:.0f} MB / {g.memoryTotal:.0f} MB') for g in GPUtil.getGPUs()]

GPU 0: 285 MB / 6144 MB


[None]

In [43]:
config = dict()
config['vocab_dim'] = len(word_to_ind)
config['output_dim'] = max_length_sentence * num_classes
config['emb_dim'] = 100
config['hidden_dim'] = 20
config['num_layers'] = 5
config['bidirectional'] = False
config['p'] = 0.7

model = RNN_1(**config)
_ = model.to(device)

In [46]:
from torchinfo import summary


summary(model, input_size=(1, config['output_dim']), dtypes=[torch.long])  # batch_size=1, seq_len=12

Layer (type:depth-idx)                   Output Shape              Param #
RNN_1                                    [1, 18, 15]               --
├─Embedding: 1-1                         [1, 270, 100]             4,351,500
├─LSTM: 1-2                              [1, 270, 20]              23,200
├─Linear: 1-3                            [1, 270]                  54,270
Total params: 4,428,970
Trainable params: 4,428,970
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 10.67
Input size (MB): 0.00
Forward/backward pass size (MB): 0.26
Params size (MB): 17.72
Estimated Total Size (MB): 17.98

## Смотрим на предскзаания без обучения

In [47]:
testing_on_test_sample(model, dataset_test_pt)

              precision    recall  f1-score   support

           0       0.05      0.02      0.03    369419
           1       0.28      0.04      0.07    736140
           2       0.10      0.09      0.10    324826
           3       0.09      0.07      0.08    245927
           4       0.07      0.05      0.06    195355
           5       0.07      0.00      0.00    317368
           6       0.24      0.00      0.00    381870
           7       0.02      0.17      0.03     41171
           8       0.00      0.00      0.00     68653
           9       0.02      0.04      0.03     93678
          10       0.02      0.08      0.03     49682
          11       0.01      0.04      0.01     46287
          12       0.02      0.26      0.03     51295
          13       0.01      0.01      0.01     28854
          14       0.00      0.04      0.01     19565
          15       0.05      0.04      0.04     29404
          16       0.00      0.01      0.00       431
          17       0.00    

In [48]:
loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam

writer = SummaryWriter(log_dir = 'model-lstm-1')
call = callback(writer, dataset_test_pt, loss_function, delimeter = 100)

trainer(count_of_epoch=5, 
        batch_size=64, 
        dataset=dataset_train_pt,
        model=model, 
        loss_function=loss_function,
        optimizer = optimizer,
        lr=0.001,
        callback=call)

epoch:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/12500 [00:00<?, ?it/s]

  0%|          | 0/12500 [00:00<?, ?it/s]

  0%|          | 0/12500 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [49]:
import os
import zipfile
from IPython.display import FileLink

folder_to_zip = '/kaggle/working/model-lstm-1'
zip_name = 'model-lstm-1.zip'

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(folder_to_zip):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, start=folder_to_zip)
            zipf.write(file_path, arcname=arcname)

# Создаём ссылку для скачивания архива
FileLink(zip_name)

/home/sasha/Documents/CourseMIPT/ML_2026_Voronzov/HW_part_1_task_2/model-lstm-1.zip

In [50]:
testing_on_test_sample(model, dataset_test_pt)

              precision    recall  f1-score   support

           0       0.57      0.84      0.68    369419
           1       0.29      0.73      0.41    736140
           2       0.38      0.18      0.25    324826
           3       0.45      0.04      0.07    245927
           4       0.29      0.03      0.05    195355
           5       0.34      0.04      0.07    317368
           6       0.33      0.26      0.29    381870
           7       0.00      0.00      0.00     41171
           8       0.21      0.00      0.00     68653
           9       0.39      0.13      0.19     93678
          10       0.08      0.00      0.00     49682
          11       0.15      0.01      0.02     46287
          12       0.23      0.02      0.04     51295
          13       0.13      0.00      0.00     28854
          14       0.00      0.00      0.00     19565
          15       0.42      0.52      0.46     29404
          16       0.00      0.00      0.00       431
          17       0.00    

In [ ]:
import GPUtil; [print(f'GPU {g.id}: {g.memoryUsed:.0f} MB / {g.memoryTotal:.0f} MB') for g in GPUtil.getGPUs()]
del model
del loss_function
del optimizer
del writer
del call
import torch
torch.cuda.empty_cache()
import GPUtil; [print(f'GPU {g.id}: {g.memoryUsed:.0f} MB / {g.memoryTotal:.0f} MB') for g in GPUtil.getGPUs()]

### Попробуем построить модель работающею с lstm_out (ну не с последними h_n c_n) а со всеми

In [14]:
class RNN_2(torch.nn.Module):
    @property
    def device(self):
        return next(self.parameters()).device

    def __init__(self, vocab_dim, num_tags, max_len, emb_dim=10, hidden_dim=10,
                 num_layers=3, bidirectional=False, p=0.7, use_batchnorm=False):
        super(RNN_2, self).__init__()
        self.num_tags = num_tags
        self.max_len = max_len
        self.vocab_dim = vocab_dim
        self.embedding = torch.nn.Embedding(vocab_dim, emb_dim, padding_idx=0)
        self.encoder = torch.nn.LSTM(emb_dim, hidden_dim, num_layers,
                                     bidirectional=bidirectional,
                                     batch_first=True, dropout=p)
        lstm_out_dim = hidden_dim * (2 if bidirectional else 1)
        self.use_batchnorm = use_batchnorm
        if use_batchnorm:
            self.batchnorm = torch.nn.BatchNorm1d(lstm_out_dim)
        self.linear = torch.nn.Linear(lstm_out_dim, num_tags)

    def forward(self, x):
        emb = self.embedding(x)
        lstm_out, _ = self.encoder(emb)
        if self.use_batchnorm:
            lstm_out = lstm_out.transpose(1, 2)
            lstm_out = self.batchnorm(lstm_out)
            lstm_out = lstm_out.transpose(1, 2)
        logits = self.linear(lstm_out)

        return logits.permute(0, 2, 1)

In [ ]:
config = {
    'vocab_dim': len(word_to_ind),
    'num_tags': num_classes,
    'max_len': max_length_sentence,
    'emb_dim': 150,
    'hidden_dim': 128,
    'num_layers': 2,
    'bidirectional': True,
    'p': 0.7,
    'use_batchnorm': False,
}

model = RNN_2(**config)
_ = model.to(device)

summary(model, input_size=(1, num_classes * max_length_sentence), dtypes=[torch.long])  # batch_size=1, seq_len=12

Layer (type:depth-idx)                   Output Shape              Param #
RNN_2                                    [1, 18, 270]              --
├─Embedding: 1-1                         [1, 270, 150]             768,150
├─LSTM: 1-2                              [1, 270, 256]             681,984
├─Linear: 1-3                            [1, 270, 18]              4,626
Total params: 1,454,760
Trainable params: 1,454,760
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 184.91
Input size (MB): 0.00
Forward/backward pass size (MB): 0.92
Params size (MB): 5.82
Estimated Total Size (MB): 6.74

In [ ]:
testing_on_test_sample(model, dataset_test_pt)

              precision    recall  f1-score   support

           0       0.00      0.00      0.00     18485
           1       0.26      0.48      0.33     36219
           2       0.00      0.00      0.00     15755
           3       0.00      0.00      0.00     12027
           4       0.00      0.00      0.00      9609
           5       0.00      0.00      0.00     16314
           6       0.00      0.00      0.00     19122
           7       0.01      0.37      0.02      2185
           8       0.00      0.00      0.00      3652
           9       0.00      0.00      0.00      4995
          10       0.00      0.00      0.00      2636
          11       0.00      0.00      0.00      2561
          12       0.06      0.00      0.01      2411
          13       0.00      0.00      0.00      1725
          14       0.00      0.00      0.00       983
          15       0.00      0.00      0.00      1282
          16       0.00      0.30      0.00        33
          17       0.00    

In [ ]:
loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam

writer = SummaryWriter(log_dir = '/kaggle/working/model-lstm-2')
call = callback(writer, dataset_test_pt, loss_function, delimeter = 100)

trainer(count_of_epoch=5, 
        batch_size=64, 
        dataset=dataset_train_pt,
        model=model, 
        loss_function=loss_function,
        optimizer = optimizer,
        lr=0.001,
        callback=call)

epoch:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

In [ ]:
testing_on_test_sample(model, dataset_test_pt)

              precision    recall  f1-score   support

           0       0.61      0.79      0.69     18485
           1       0.28      0.77      0.41     36219
           2       0.38      0.17      0.24     15755
           3       0.40      0.03      0.06     12027
           4       0.28      0.04      0.07      9609
           5       0.30      0.03      0.06     16314
           6       0.34      0.23      0.27     19122
           7       0.00      0.00      0.00      2185
           8       0.00      0.00      0.00      3652
           9       0.34      0.11      0.16      4995
          10       0.00      0.00      0.00      2636
          11       0.56      0.00      0.00      2561
          12       0.34      0.01      0.01      2411
          13       0.00      0.00      0.00      1725
          14       0.00      0.00      0.00       983
          15       0.40      0.45      0.42      1282
          16       0.00      0.00      0.00        33
          17       0.00    

### Вывод: очень плохие результаты, стоит попробывать предобученый токенизатор

In [19]:
import psutil

# Получаем статистику памяти
mem = psutil.virtual_memory()

print(f"Всего: {mem.total / (1024**3):.2f} GB")
print(f"Доступно: {mem.available / (1024**3):.2f} GB")
print(f"Используется: {mem.used / (1024**3):.2f} GB")
print(f"Процент использования: {mem.percent}%")

Всего: 15.43 GB
Доступно: 7.95 GB
Используется: 7.49 GB
Процент использования: 48.5%


In [20]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/LaBSE")

In [21]:
train_texts = [item[0] for item in dataset_train]
test_texts  = [item[0] for item in dataset_test]

train_data_sent = tokenizer(train_texts, padding=True, truncation=True, max_length=max_length_sentence, return_tensors="pt")['input_ids']
test_data_sent  = tokenizer(test_texts, padding=True, truncation=True, max_length=max_length_sentence, return_tensors="pt")['input_ids']

train_y = []
test_y  = []

for item in dataset_train:
    if (len(item[1]) < max_length_sentence):
        item[1].extend([0] * (max_length_sentence - len(item[1]))) # добовляем падинг!
    train_y.append(item[1][:max_length_sentence])

for item in dataset_test:
    if (len(item[1]) < max_length_sentence):
        item[1].extend([0] * (max_length_sentence - len(item[1]))) # добовляем падинг!
    test_y.append(item[1][:max_length_sentence])

dataset_train_pt = torch.utils.data.TensorDataset(
    train_data_sent, torch.tensor(train_y))
dataset_test_pt = torch.utils.data.TensorDataset(
    test_data_sent, torch.tensor(test_y))

In [22]:
all_ids = []
for text in train_texts:
    # Токенизируем без паддинга, чтобы получить реальные токены
    ids = tokenizer.encode(text, add_special_tokens=True)  # учитывает [CLS] и [SEP]
    all_ids.extend(ids)

for text in test_texts:
    # Токенизируем без паддинга, чтобы получить реальные токены
    ids = tokenizer.encode(text, add_special_tokens=True)  # учитывает [CLS] и [SEP]
    all_ids.extend(ids)


# Уникальные токены
unique_tokens = set(all_ids)
print(f"Всего токенов (с повторениями): {len(all_ids)}")
print(f"Уникальных токенов: {len(unique_tokens)}")

Всего токенов (с повторениями): 27940726
Уникальных токенов: 62387


In [23]:
import psutil

# Получаем статистику памяти
mem = psutil.virtual_memory()

print(f"Всего: {mem.total / (1024**3):.2f} GB")
print(f"Доступно: {mem.available / (1024**3):.2f} GB")
print(f"Используется: {mem.used / (1024**3):.2f} GB")
print(f"Процент использования: {mem.percent}%")

Всего: 15.43 GB
Доступно: 2.19 GB
Используется: 13.24 GB
Процент использования: 85.8%


In [ ]:
config = {
    'vocab_dim': len(all_ids),
    'num_tags': num_classes,
    'max_len': max_length_sentence,
    'emb_dim': 150,
    'hidden_dim': 128,
    'num_layers': 2,
    'bidirectional': True,
    'p': 0.7,
    'use_batchnorm': False,
}


model = RNN_2(**config)
_ = model.to(device)

summary(model, input_size=(1, num_classes * max_length_sentence), dtypes=[torch.long])  # batch_size=1, seq_len=12

In [ ]:
loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam

writer = SummaryWriter(log_dir = '/kaggle/working/model-lstm-3')
call = callback(writer, dataset_test_pt, loss_function, delimeter = 100)

trainer(count_of_epoch=5, 
        batch_size=64, 
        dataset=dataset_train_pt,
        model=model, 
        loss_function=loss_function,
        optimizer = optimizer,
        lr=0.001,
        callback=call)

epoch:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

In [ ]:
testing_on_test_sample(model, dataset_test_pt)

              precision    recall  f1-score   support

           0       0.62      0.86      0.72     18485
           1       0.51      0.73      0.60     36219
           2       0.78      0.61      0.68     15755
           3       0.71      0.37      0.48     12027
           4       0.70      0.50      0.58      9609
           5       0.68      0.55      0.61     16314
           6       0.59      0.58      0.59     19122
           7       0.81      0.50      0.62      2185
           8       0.79      0.46      0.58      3652
           9       0.86      0.63      0.73      4995
          10       0.62      0.39      0.48      2636
          11       0.84      0.70      0.76      2561
          12       0.64      0.36      0.46      2411
          13       0.75      0.52      0.61      1725
          14       0.74      0.62      0.67       983
          15       0.79      0.40      0.53      1282
          16       0.00      0.00      0.00        33
          17       0.00    

## Мы выбели гараздо более лучший скор!

In [ ]:
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ru.300.bin.gz
!gunzip cc.ru.300.bin.gz

--2026-02-24 15:57:03--  https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ru.300.bin.gz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 99.84.118.67, 99.84.118.117, 99.84.118.60, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|99.84.118.67|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4496459151 (4.2G) [application/octet-stream]
Saving to: ‘cc.ru.300.bin.gz’

cc.ru.300.bin.gz    100%[===================>]   4.19G   301MB/s    in 14s     

2026-02-24 15:57:17 (306 MB/s) - ‘cc.ru.300.bin.gz’ saved [4496459151/4496459151]



In [ ]:
import fasttext

# Загрузка модели (потребуется время и место на диске)
ft = fasttext.load_model('cc.ru.300.bin')

In [ ]:
word_to_ind = dict()
matrix_fasttext = []
for i, w in enumerate(tqdm(ft.get_words(on_unicode_error='replace'))):
    v = ft.get_word_vector(w)
    if w not in word_to_ind:
        word_to_ind[w] = i
        matrix_fasttext.append(v)
for w in ['[PAD]', '[UNK]', '[CLS]', '[SEP]']:
    word_to_ind[w] = word_to_ind.__len__()
    matrix_fasttext.append(np.zeros_like(matrix_fasttext[-1]))

tokenizer = Tokenizer(word_to_ind, RegexpTokenizer('[a-zA-Z]+|[^\w\s]|\d+'))

train_texts = [item[0] for item in dataset_train]
test_texts  = [item[0] for item in dataset_test]

train_data_sent = tokenizer(train_texts)
test_data_sent  = tokenizer(test_texts)

train_y = []
test_y  = []

for item in dataset_train:
    if (len(item[1]) < max_length_sentence):
        item[1].extend([0] * (max_length_sentence - len(item[1]))) # добовляем падинг!
    train_y.append(item[1][:max_length_sentence])

for item in dataset_test:
    if (len(item[1]) < max_length_sentence):
        item[1].extend([0] * (max_length_sentence - len(item[1]))) # добовляем падинг!
    test_y.append(item[1][:max_length_sentence])

dataset_train_pt = torch.utils.data.TensorDataset(
    train_data_sent, torch.tensor(train_y))
dataset_test_pt = torch.utils.data.TensorDataset(
    test_data_sent, torch.tensor(test_y))

  0%|          | 0/2000000 [00:00<?, ?it/s]

In [ ]:
config = {
    'vocab_dim': len(word_to_ind),
    'num_tags': num_classes,
    'max_len': max_length_sentence,
    'emb_dim': 300,
    'hidden_dim': 128,
    'num_layers': 3,
    'bidirectional': True,
    'p': 0.7,
    'use_batchnorm': False,
}


model = RNN_2(**config)
_ = model.to(device)

model.embedding.weight.data.copy_(torch.tensor(matrix_fasttext))
for param in model.embedding.parameters():
    param.requires_grad = False
model.to(device)

summary(model, input_size=(1, num_classes * max_length_sentence), dtypes=[torch.long])

Layer (type:depth-idx)                   Output Shape              Param #
RNN_2                                    [1, 18, 270]              --
├─Embedding: 1-1                         [1, 270, 300]             (600,001,200)
├─LSTM: 1-2                              [1, 270, 256]             1,230,848
├─Linear: 1-3                            [1, 270, 18]              4,626
Total params: 601,236,674
Trainable params: 1,235,474
Non-trainable params: 600,001,200
Total mult-adds (Units.MEGABYTES): 932.33
Input size (MB): 0.00
Forward/backward pass size (MB): 1.24
Params size (MB): 2404.95
Estimated Total Size (MB): 2406.19

In [ ]:
loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam

writer = SummaryWriter(log_dir = '/kaggle/working/model-lstm-4')
call = callback(writer, dataset_test_pt, loss_function, delimeter = 100)

trainer(count_of_epoch=5, 
        batch_size=64, 
        dataset=dataset_train_pt,
        model=model, 
        loss_function=loss_function,
        optimizer = optimizer,
        lr=0.001,
        callback=call)

epoch:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

In [ ]:
testing_on_test_sample(model, dataset_test_pt)

              precision    recall  f1-score   support

           0       0.53      0.88      0.66     18485
           1       0.28      0.74      0.41     36219
           2       0.38      0.16      0.23     15755
           3       0.47      0.03      0.05     12027
           4       0.44      0.03      0.05      9609
           5       0.33      0.03      0.06     16314
           6       0.33      0.22      0.26     19122
           7       0.00      0.00      0.00      2185
           8       0.00      0.00      0.00      3652
           9       0.35      0.11      0.16      4995
          10       0.00      0.00      0.00      2636
          11       0.50      0.00      0.00      2561
          12       0.00      0.00      0.00      2411
          13       0.00      0.00      0.00      1725
          14       0.00      0.00      0.00       983
          15       0.40      0.55      0.46      1282
          16       0.00      0.00      0.00        33
          17       0.00    

In [ ]:
del matrix_fasttext

### Итог очень плохой, есть подозрение что проблема в большой размерности. И в том что мы не можем использовать весь датасет...

## Попробуем выбить максимальный скор с помощью 3ей модели

In [ ]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/LaBSE")

In [ ]:
train_texts = [item[0] for item in dataset_train]
test_texts  = [item[0] for item in dataset_test]

train_data_sent = tokenizer(train_texts, padding=True, truncation=True, max_length=max_length_sentence, return_tensors="pt")['input_ids']
test_data_sent  = tokenizer(test_texts, padding=True, truncation=True, max_length=max_length_sentence, return_tensors="pt")['input_ids']

train_y = []
test_y  = []

for item in dataset_train:
    if (len(item[1]) < max_length_sentence):
        item[1].extend([0] * (max_length_sentence - len(item[1]))) # добовляем падинг!
    train_y.append(item[1][:max_length_sentence])

for item in dataset_test:
    if (len(item[1]) < max_length_sentence):
        item[1].extend([0] * (max_length_sentence - len(item[1]))) # добовляем падинг!
    test_y.append(item[1][:max_length_sentence])

dataset_train_pt = torch.utils.data.TensorDataset(
    train_data_sent, torch.tensor(train_y))
dataset_test_pt = torch.utils.data.TensorDataset(
    test_data_sent, torch.tensor(test_y))

In [ ]:
all_ids = []
for text in train_texts:
    # Токенизируем без паддинга, чтобы получить реальные токены
    ids = tokenizer.encode(text, add_special_tokens=True)  # учитывает [CLS] и [SEP]
    all_ids.extend(ids)

for text in test_texts:
    # Токенизируем без паддинга, чтобы получить реальные токены
    ids = tokenizer.encode(text, add_special_tokens=True)  # учитывает [CLS] и [SEP]
    all_ids.extend(ids)

### Пробуем batchnorm

In [ ]:
config = {
    'vocab_dim': len(all_ids),
    'num_tags': num_classes,
    'max_len': max_length_sentence,
    'emb_dim': 150,
    'hidden_dim': 128,
    'num_layers': 2,
    'bidirectional': True,
    'p': 0.7,
    'use_batchnorm': True,
}


model = RNN_2(**config)
_ = model.to(device)

summary(model, input_size=(1, num_classes * max_length_sentence), dtypes=[torch.long])

Layer (type:depth-idx)                   Output Shape              Param #
RNN_2                                    [1, 18, 270]              --
├─Embedding: 1-1                         [1, 270, 150]             208,729,200
├─LSTM: 1-2                              [1, 270, 256]             681,984
├─BatchNorm1d: 1-3                       [1, 256, 270]             512
├─Linear: 1-4                            [1, 270, 18]              4,626
Total params: 209,416,322
Trainable params: 209,416,322
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 392.87
Input size (MB): 0.00
Forward/backward pass size (MB): 1.47
Params size (MB): 837.67
Estimated Total Size (MB): 839.14

In [ ]:
loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam

writer = SummaryWriter(log_dir = '/kaggle/working/model-lstm-5')
call = callback(writer, dataset_test_pt, loss_function, delimeter = 100)

trainer(count_of_epoch=5, 
        batch_size=64, 
        dataset=dataset_train_pt,
        model=model, 
        loss_function=loss_function,
        optimizer = optimizer,
        lr=0.001,
        callback=call)

epoch:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

In [ ]:
testing_on_test_sample(model, dataset_test_pt)

              precision    recall  f1-score   support

           0       0.69      0.82      0.75     18485
           1       0.49      0.79      0.60     36219
           2       0.82      0.60      0.69     15755
           3       0.78      0.36      0.49     12027
           4       0.71      0.50      0.59      9609
           5       0.68      0.58      0.62     16314
           6       0.66      0.55      0.60     19122
           7       0.78      0.53      0.63      2185
           8       0.79      0.50      0.61      3652
           9       0.81      0.67      0.73      4995
          10       0.61      0.42      0.49      2636
          11       0.83      0.71      0.76      2561
          12       0.75      0.36      0.48      2411
          13       0.77      0.52      0.62      1725
          14       0.77      0.63      0.69       983
          15       0.76      0.42      0.54      1282
          16       0.00      0.00      0.00        33
          17       0.00    

In [ ]:
import torch
import gc
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

del model, optimizer, loss_function
gc.collect()
torch.cuda.empty_cache()
print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   5527 MiB |   7924 MiB |  13949 GiB |  13944 GiB |
|       from large pool |   5525 MiB |   7913 MiB |  13710 GiB |  13705 GiB |
|       from small pool |      2 MiB |     16 MiB |    239 GiB |    239 GiB |
|---------------------------------------------------------------------------|
| Active memory         |   5527 MiB |   7924 MiB |  13949 GiB |  13944 GiB |
|       from large pool |   5525 MiB |   7913 MiB |  13710 GiB |

### batchnorm выбил лучший скор будем использовать его! (p.s датачет большой берём только часть, чаще всего он выбивает лучше скор)

## Попробуем покрутить число слаёв размер скрутого слоя размерность ембединга

In [ ]:
config = {
    'vocab_dim': len(all_ids),
    'num_tags': num_classes,
    'max_len': max_length_sentence,
    'emb_dim': 200,
    'hidden_dim': 64,
    'num_layers': 5,
    'bidirectional': True,
    'p': 0.7,
    'use_batchnorm': True,
}


model = RNN_2(**config)
_ = model.to(device)

summary(model, input_size=(1, num_classes * max_length_sentence), dtypes=[torch.long])

Layer (type:depth-idx)                   Output Shape              Param #
RNN_2                                    [1, 18, 270]              --
├─Embedding: 1-1                         [1, 270, 200]             278,305,600
├─LSTM: 1-2                              [1, 270, 128]             533,504
├─BatchNorm1d: 1-3                       [1, 128, 270]             256
├─Linear: 1-4                            [1, 270, 18]              2,322
Total params: 278,841,682
Trainable params: 278,841,682
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 422.35
Input size (MB): 0.00
Forward/backward pass size (MB): 1.02
Params size (MB): 1115.37
Estimated Total Size (MB): 1116.39

In [ ]:
loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam

writer = SummaryWriter(log_dir = '/kaggle/working/model-lstm-6')
call = callback(writer, dataset_test_pt, loss_function, delimeter = 100)

trainer(count_of_epoch=5, 
        batch_size=64, 
        dataset=dataset_train_pt,
        model=model, 
        loss_function=loss_function,
        optimizer = optimizer,
        lr=0.001,
        callback=call)

epoch:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

In [ ]:
testing_on_test_sample(model, dataset_test_pt)

              precision    recall  f1-score   support

           0       0.66      0.82      0.73     18485
           1       0.45      0.76      0.57     36219
           2       0.75      0.61      0.67     15755
           3       0.66      0.23      0.34     12027
           4       0.67      0.37      0.48      9609
           5       0.60      0.50      0.55     16314
           6       0.60      0.57      0.58     19122
           7       0.77      0.40      0.53      2185
           8       0.83      0.27      0.41      3652
           9       0.78      0.65      0.71      4995
          10       0.49      0.34      0.40      2636
          11       0.82      0.69      0.75      2561
          12       0.74      0.06      0.12      2411
          13       0.49      0.32      0.39      1725
          14       0.58      0.38      0.46       983
          15       0.70      0.23      0.34      1282
          16       0.00      0.00      0.00        33
          17       0.00    

In [ ]:
import torch
import gc
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

del model, optimizer, loss_function
gc.collect()
torch.cuda.empty_cache()
print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   7655 MiB |  10848 MiB |  21847 GiB |  21839 GiB |
|       from large pool |   7653 MiB |  10839 MiB |  21551 GiB |  21544 GiB |
|       from small pool |      2 MiB |     16 MiB |    295 GiB |    295 GiB |
|---------------------------------------------------------------------------|
| Active memory         |   7655 MiB |  10848 MiB |  21847 GiB |  21839 GiB |
|       from large pool |   7653 MiB |  10839 MiB |  21551 GiB |

In [ ]:
config = {
    'vocab_dim': len(all_ids),
    'num_tags': num_classes,
    'max_len': max_length_sentence,
    'emb_dim': 200,
    'hidden_dim': 150,
    'num_layers': 3,
    'bidirectional': True,
    'p': 0.7,
    'use_batchnorm': True,
}


model = RNN_2(**config)
_ = model.to(device)

summary(model, input_size=(1, num_classes * max_length_sentence), dtypes=[torch.long])

Layer (type:depth-idx)                   Output Shape              Param #
RNN_2                                    [1, 18, 270]              --
├─Embedding: 1-1                         [1, 270, 200]             278,305,600
├─LSTM: 1-2                              [1, 270, 300]             1,507,200
├─BatchNorm1d: 1-3                       [1, 300, 270]             600
├─Linear: 1-4                            [1, 270, 18]              5,418
Total params: 279,818,818
Trainable params: 279,818,818
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 685.26
Input size (MB): 0.00
Forward/backward pass size (MB): 1.77
Params size (MB): 1119.28
Estimated Total Size (MB): 1121.04

In [ ]:
loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam

writer = SummaryWriter(log_dir = '/kaggle/working/model-lstm-7')
call = callback(writer, dataset_test_pt, loss_function, delimeter = 100)

trainer(count_of_epoch=5, 
        batch_size=64, 
        dataset=dataset_train_pt,
        model=model, 
        loss_function=loss_function,
        optimizer = optimizer,
        lr=0.001,
        callback=call)

epoch:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

In [ ]:
testing_on_test_sample(model, dataset_test_pt)

              precision    recall  f1-score   support

           0       0.64      0.85      0.73     18485
           1       0.55      0.72      0.62     36219
           2       0.84      0.63      0.72     15755
           3       0.69      0.48      0.57     12027
           4       0.69      0.57      0.63      9609
           5       0.71      0.61      0.66     16314
           6       0.59      0.61      0.60     19122
           7       0.80      0.58      0.67      2185
           8       0.82      0.52      0.64      3652
           9       0.85      0.68      0.76      4995
          10       0.68      0.43      0.53      2636
          11       0.85      0.74      0.79      2561
          12       0.77      0.40      0.53      2411
          13       0.77      0.56      0.65      1725
          14       0.78      0.68      0.72       983
          15       0.75      0.45      0.56      1282
          16       1.00      0.03      0.06        33
          17       0.00    

In [ ]:
config = {
    'vocab_dim': len(all_ids),
    'num_tags': num_classes,
    'max_len': max_length_sentence,
    'emb_dim': 100,
    'hidden_dim': 200,
    'num_layers': 3,
    'bidirectional': True,
    'p': 0.7,
    'use_batchnorm': True,
}


model = RNN_2(**config)
_ = model.to(device)

summary(model, input_size=(1, num_classes * max_length_sentence), dtypes=[torch.long])

Layer (type:depth-idx)                   Output Shape              Param #
RNN_2                                    [1, 18, 270]              --
├─Embedding: 1-1                         [1, 270, 100]             139,152,800
├─LSTM: 1-2                              [1, 270, 400]             2,409,600
├─BatchNorm1d: 1-3                       [1, 400, 270]             800
├─Linear: 1-4                            [1, 270, 18]              7,218
Total params: 141,570,418
Trainable params: 141,570,418
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 789.75
Input size (MB): 0.00
Forward/backward pass size (MB): 1.98
Params size (MB): 566.28
Estimated Total Size (MB): 568.27

In [ ]:
loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam

writer = SummaryWriter(log_dir = '/kaggle/working/model-lstm-8')
call = callback(writer, dataset_test_pt, loss_function, delimeter = 100)

trainer(count_of_epoch=5, 
        batch_size=64, 
        dataset=dataset_train_pt,
        model=model, 
        loss_function=loss_function,
        optimizer = optimizer,
        lr=0.001,
        callback=call)

epoch:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/625 [00:00<?, ?it/s]

In [ ]:
testing_on_test_sample(model, dataset_test_pt)

              precision    recall  f1-score   support

           0       0.69      0.81      0.75     18485
           1       0.49      0.76      0.59     36219
           2       0.82      0.62      0.71     15755
           3       0.68      0.41      0.51     12027
           4       0.72      0.52      0.60      9609
           5       0.70      0.58      0.64     16314
           6       0.65      0.57      0.61     19122
           7       0.82      0.52      0.64      2185
           8       0.82      0.48      0.61      3652
           9       0.83      0.68      0.75      4995
          10       0.67      0.41      0.51      2636
          11       0.87      0.71      0.78      2561
          12       0.71      0.39      0.50      2411
          13       0.78      0.55      0.64      1725
          14       0.74      0.67      0.70       983
          15       0.71      0.47      0.56      1282
          16       0.00      0.00      0.00        33
          17       0.00    

In [ ]:
import os
import zipfile
from IPython.display import FileLink

folder_to_zip = '/kaggle/working/model-lstm-8'
zip_name = 'model-lstm-8.zip'

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(folder_to_zip):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, start=folder_to_zip)
            zipf.write(file_path, arcname=arcname)

# Создаём ссылку для скачивания архива
FileLink(zip_name)

/kaggle/working/model-lstm-8.zip

## Лучше всего себя показала данная модель:
config = {
    'vocab_dim': len(all_ids),
    'num_tags': num_classes,
    'max_len': max_length_sentence,
    'emb_dim': 200,
    'hidden_dim': 150,
    'num_layers': 3,
    'bidirectional': True,
    'p': 0.7,
    'use_batchnorm': True,
}
